In [1]:
import json
import re
import time
from datetime import date, timedelta
import os
import polars as pl
import requests

from bs4 import BeautifulSoup

##### Web scraping

In [2]:
BASE = "https://anc.apm.activecommunities.com/chicagoparkdistrict"
SEARCH_PAGE = BASE + "/activity/search?onlineSiteId=0&viewMode=list"
API_URL = BASE + "/rest/activities/list?locale=en-US"
DETAIL_URL_TEMPLATE = BASE + "/rest/activity/detail/{activity_id}?locale=en-US"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Content-Type": "application/json",
    "Referer": SEARCH_PAGE,
}

PAGE_SIZE = 20            
DELAY_TIME = 1  

In [9]:
# Parameters 

date_after = "2015-12-31"
date_before = ""
label = f"{date_after}_to_{date_before or 'now'}"

In [ ]:
# Scraping helper functions

# Handle cookies and CSRF token
def make_session():
    session = requests.Session()
    session.headers.update(HEADERS)
    resp = session.get(SEARCH_PAGE)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    for script in soup.find_all("script"):
        text = script.string or ""
        match = re.search(r'__csrfToken\s*=\s*"([^"]+)"', text)
        if match:
            session.headers["X-CSRF-Token"] = match.group(1)
            break
    return session

# Get one page of search results as JSON 
def fetch_page(session: requests.Session, page_number: int,
               date_after: str, date_before: str, keyword: str = ""):
    body = {
        "activity_search_pattern": {
            "skills": [],
            "time_after_str": "",
            "days_of_week": "0000000", 
            "activity_select_param": 2,
            "center_ids": [],
            "time_before_str": "",
            "open_spots": None,
            "activity_id": None,
            "activity_category_ids": [],
            "date_before": "",
            "min_age": None,
            "date_before": date_before,
            "date_after": date_after,   
            "activity_type_ids": [],
            "site_ids": [],
            "for_map": False,
            "geographic_area_ids": [],
            "season_ids": [],
            "activity_department_ids": [],
            "activity_other_category_ids": [],
            "child_season_ids": [],
            "activity_keyword": keyword,
            "instructor_ids": [],
            "max_age": None,
            "custom_price_from": "",
            "custom_price_to": "",
        },
        "activity_transfer_pattern": {},
    }
    page_info = {"order_by": "", "page_number": page_number,
                 "total_records_per_page": PAGE_SIZE}

    resp = session.post(API_URL, json=body,
                        headers={"page_info": json.dumps(page_info)})
    resp.raise_for_status()
    return resp.json()

def extract(item: dict):
    return {
        "activity_name": item.get("name"),
        "park": (item.get("location") or {}).get("label"),
        "ages": item.get("ages"),
        "age_description": item.get("age_description"),
        "date_start": item.get("date_range_start"),
        "date_end": item.get("date_range_end"),
        "date_range": item.get("date_range"),
        "time": item.get("time_range"),
        "days_of_week": item.get("days_of_week"),
        "category": item.get("category"),
        "fee": (item.get("fee") or {}).get("label"),
        "status": (item.get("urgent_message") or {}).get("status_description"),
        "activity_id": item.get("id"),
        "detail_url": item.get("detail_url"),
    }

# Go through all the pages and scrape data
def scrape(date_before: str, date_after: str, 
            max_pages: int | None = None, keyword: str = ""):
    
    session = make_session()

    data = fetch_page(session, page_number=1, 
                    date_before=date_before, date_after=date_after, 
                    keyword=keyword)
    items = data.get("body", {}).get("activity_items", [])
    assert items, f"No activity_items found. Response keys: {list(data.keys())}"

    page_meta = data.get("headers", {}).get("page_info", {})
    total_pages = int(page_meta.get("total_page", 1))

    rows = [extract(i) for i in items]

    last_page = total_pages if max_pages is None else min(total_pages, max_pages)
    for page in range(2, last_page + 1):
        time.sleep(DELAY_TIME)
        data = fetch_page(session, page_number=page, date_before=date_before, 
                        date_after=date_after, keyword=keyword)
        page_items = data.get("body", {}).get("activity_items", [])
        if not page_items:
            break
        rows.extend(extract(i) for i in page_items)

    
    df = pl.DataFrame(rows, infer_schema_length=None)
    label = f"{date_after}_to_{date_before or 'now'}"
    df.write_csv(f"data/cpd_activities_{label}.csv")
    df.write_parquet(f"data/cpd_activities_{label}.parquet")

    return df, session

# Get more detailed info for one specified activity
def fetch_activity_detail(session: requests.Session, activity_id: int):
    resp = session.get(DETAIL_URL_TEMPLATE.format(activity_id=activity_id))
    resp.raise_for_status()
    detail = resp.json().get("body", {}).get("detail", {})
    centers = detail.get("centers") or [{}]
    return {
        "activity_id": detail.get("activity_id"),
        "season_name": detail.get("season_name"),
        "term_name": detail.get("term_name"),
        "sub_category": detail.get("sub_category"),
        "activity_type": detail.get("activity_type"),
        "allowed_gender": detail.get("allowed_gender"),
        "sessions": (detail.get("other_info") or {}).get("sessions"),
        "department": (detail.get("other_info") or {}).get("department"),
        "center_address": centers[0].get("address1"),
        "center_zip": centers[0].get("zip_code"),
        "latitude": centers[0].get("latitude"),
        "longitude": centers[0].get("longitude"),
    }

In [ ]:
from_2016_to_now, session_2016_to_now = scrape(date_after = date_after, 
                                               date_before = date_before,
                                               max_pages = None, 
                                               keyword = "") 

Total: 115374 records across 5769 pages
First record starts: '2023-06-25'
Page 2/5769: 40 activities so far
Page 3/5769: 60 activities so far
Page 4/5769: 80 activities so far
Page 5/5769: 100 activities so far
Page 6/5769: 120 activities so far
Page 7/5769: 140 activities so far
Page 8/5769: 160 activities so far
Page 9/5769: 180 activities so far
Page 10/5769: 200 activities so far
Page 11/5769: 220 activities so far
Page 12/5769: 240 activities so far
Page 13/5769: 260 activities so far
Page 14/5769: 280 activities so far
Page 15/5769: 300 activities so far
Page 16/5769: 320 activities so far
Page 17/5769: 340 activities so far
Page 18/5769: 360 activities so far
Page 19/5769: 380 activities so far
Page 20/5769: 400 activities so far
Page 21/5769: 420 activities so far
Page 22/5769: 440 activities so far
Page 23/5769: 460 activities so far
Page 24/5769: 480 activities so far
Page 25/5769: 500 activities so far
Page 26/5769: 520 activities so far
Page 27/5769: 540 activities so far
P

In [4]:
# Derive activity seasons

def derive_seasons(df):
    df = df.with_columns(
        pl.col("date_start").str.to_date(strict=False).alias("date_start_parsed")
        ).with_columns(
            season_inferred=(
                pl.when(pl.col("date_start_parsed").dt.month().is_in([3, 4, 5]))
                .then(pl.lit("Spring"))
                .when(pl.col("date_start_parsed").dt.month().is_in([6, 7, 8]))
                .then(pl.lit("Summer"))
                .when(pl.col("date_start_parsed").dt.month().is_in([9, 10, 11]))
                .then(pl.lit("Fall"))
                .otherwise(pl.lit("Winter"))
            ),
            season_year=pl.col("date_start_parsed").dt.year(),
        )
    
    return df 

# get park lat/long from detailed activity pages

def find_park_locations(session, df):

    PARK_CACHE = "data/park_locations.csv"

    if os.path.exists(PARK_CACHE):
        park_lookup = pl.read_csv(PARK_CACHE, infer_schema_length=None,
                                schema_overrides={"center_zip": pl.Utf8})
        cached_parks = set(park_lookup["park"].to_list())
    else:
        park_lookup = None
        cached_parks = set()

    one_per_park = (
        df.filter(~pl.col("park").is_in(cached_parks))
        .unique(subset=["park"], keep="first")
        .select("park", "activity_id")
    )

    park_rows = []
    for i, (park, activity_id) in enumerate(one_per_park.iter_rows(), start=1):
        time.sleep(DELAY_TIME)
        try:
            d = fetch_activity_detail(session, activity_id)
            park_rows.append({
                "park": park,
                "center_address": d["center_address"],
                "center_zip": d["center_zip"],
                "latitude": d["latitude"],
                "longitude": d["longitude"],
            })
        except Exception as e:
            print(f"  {park} (activity {activity_id}) failed: {e}")
        if i % 25 == 0:
            print(f"  {i}/{len(one_per_park)}")

    if park_rows:
        new_parks = pl.DataFrame(park_rows, infer_schema_length=None)
        park_lookup = new_parks if park_lookup is None else pl.concat([park_lookup, new_parks])
        park_lookup.write_csv(PARK_CACHE)

    missing = park_lookup.filter(pl.col("latitude").is_null())
    if len(missing):
        print(f"{len(missing)} parks lack coordinates:", missing["park"].to_list())

    df = df.join(park_lookup, on="park", how="left")
    
    return df

In [ ]:
from_2016_to_now_df = derive_seasons(from_2016_to_now)
from_2016_to_now_df = find_park_locations(session_2016_to_now, from_2016_to_now_df)

from_2016_to_now_df.write_parquet(f"data/cpd_activities_{label}_detailed.parquet")